# Experiment 1 – Report vs Promote

Goal: explain *why* a toxicity filter (Detoxify) flags sentences that report or argue against abuse, using SHAP and LIME, and measure whether the two explainers agree. Runtime: Colab → Runtime → Change runtime type → T4 GPU.

## Cell 1: install, imports and shared helpers

In [ ]:
!pip -q install detoxify shap lime
import importlib.metadata, os, re, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, shap, lime, sklearn
from matplotlib import colors as mcolors
from detoxify import Detoxify
from lime.lime_text import LimeTextExplainer
from scipy.stats import spearmanr
print("shap", shap.__version__, "| lime", importlib.metadata.version("lime"), "| torch", torch.__version__)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
OUT = "outputs_exp1"; os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"font.size": 14, "axes.titlesize": 16, "axes.labelsize": 14})
COLORS = {"original": "#0072B2", "unbiased": "#E69F00"}   # Okabe-Ito, colour-blind safe

MODELS = {m: Detoxify(m, device=DEVICE) for m in ["original", "unbiased"]}

def proba(texts, model="original", bs=64):
    """P(toxic) for a list of strings, batched so LIME's thousands of samples don't run out of memory."""
    texts = [str(t) for t in texts]
    out = []
    for i in range(0, len(texts), bs):
        out.extend(np.atleast_1d(MODELS[model].predict(texts[i:i+bs])["toxicity"]))
    return np.array(out, dtype=float)

def logit(texts, model="original"):
    """Explain log-odds, not probability, so SHAP contributions are additive (shap-exercises notebook)."""
    p = np.clip(proba(texts, model), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def lime_fn(model="original"):
    """friday-03 pattern: list of strings -> (n, 2) array of [P(non-toxic), P(toxic)]."""
    def f(texts):
        p = proba(texts, model)
        return np.column_stack([1 - p, p])
    return f

def word_tokenizer(s, return_offsets_mapping=True):
    """Word-level tokens that keep their trailing punctuation. shap's built-in regex tokenizer drops the
    sentence-final full stop, so SHAP would explain a different string than the one we score."""
    spans = [m.span() for m in re.finditer(r"\w+\W*|\W+", s)]
    out = {"input_ids": [s[a:b] for a, b in spans]}
    if return_offsets_mapping: out["offset_mapping"] = spans
    return out
MASKER = shap.maskers.Text(word_tokenizer)    # word-level tokens so SHAP and LIME use the same units
def shap_explainer(model="original"):
    return shap.Explainer(lambda t: logit(t, model), MASKER)

def check_efficiency(sv, texts, model, tol=1e-2):
    """Efficiency/additivity axiom (W4-L01 p4, textbook p48): base value + sum of SHAP values = model output."""
    f = logit(texts, model)
    recon = np.array([float(sv[i].base_values) + sv[i].values.sum() for i in range(len(texts))])
    err = np.abs(recon - f).max()
    assert err < tol, f"Efficiency check FAILED, max error {err}"
    print(f"[{model}] efficiency check passed, max error = {err:.2e}")

def shap_word_weights(e):
    """Collapse SHAP tokens into lower-case words; sum repeated words."""
    w = {}
    for tok, v in zip(e.data, e.values):
        k = re.sub(r"\W+", "", tok).lower()
        if k: w[k] = w.get(k, 0.0) + float(v)
    return w

def lime_word_weights(text, model="original", seed=SEED, num_samples=2000):
    """LIME on one sentence. labels=(1,) so weights refer to the TOXIC class (friday-01 sign trap)."""
    expl = LimeTextExplainer(class_names=["non-toxic", "toxic"], random_state=seed)
    n = len(set(re.findall(r"\w+", text)))
    exp = expl.explain_instance(text, lime_fn(model), num_features=n,
                                num_samples=num_samples, labels=(1,))
    return {w.lower(): v for w, v in exp.as_list(label=1)}, exp

def agreement(sw, lw, k=3):
    """Krishna et al. (2022)-style agreement: rank correlation, top-1 match, top-k overlap.
    Ranks are used because SHAP is in log-odds and LIME is in probability, so raw sizes are not comparable."""
    common = sorted(set(sw) & set(lw))
    rho = spearmanr([sw[w] for w in common], [lw[w] for w in common]).correlation if len(common) > 2 else np.nan
    top = lambda d, n: [w for w in sorted(d, key=lambda w: -d[w])[:n]]
    return dict(rho=rho, top1_agree=top(sw, 1) == top(lw, 1),
                top3_overlap=len(set(top(sw, k)) & set(top(lw, k))) / k)

def highlight_png(pairs, title, path):
    """Draw a sentence with each word coloured by its attribution (red = toward toxic, blue = away).
    Colours are scaled per figure, so compare colour patterns between SHAP and LIME, not exact shades."""
    vmax = max([abs(v) for _, v in pairs] + [1e-9])
    norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax); cmap = plt.get_cmap("RdBu_r")
    fig, ax = plt.subplots(figsize=(14, 1.4)); ax.axis("off")
    fig.canvas.draw(); r = fig.canvas.get_renderer(); x = 0.005
    for w, v in pairs:
        t = ax.text(x, 0.5, w, fontsize=20, va="center", transform=ax.transAxes,
                    bbox=dict(boxstyle="round,pad=0.25", fc=cmap(norm(v)), ec="none"))
        x = t.get_window_extent(renderer=r).transformed(ax.transAxes.inverted()).x1 + 0.01
    ax.set_title(title, loc="left", fontsize=16)
    fig.savefig(path, dpi=200, bbox_inches="tight"); plt.close(fig)

## Cell 2: the sentences (do not edit)

In [ ]:
ROWS = [
 (1,"football fans","promote","Those fans are disgusting idiots and should be banned.","disgusting idiots"),
 (1,"football fans","report","The club condemned a post calling its fans disgusting idiots who should be banned.","disgusting idiots"),
 (1,"football fans","counter","Calling the fans disgusting idiots is unfair and wrong.","disgusting idiots"),
 (2,"immigrants","promote","Immigrants are worthless and we hate them here.","worthless hate"),
 (2,"immigrants","report","Police are investigating leaflets claiming immigrants are worthless.","worthless"),
 (2,"immigrants","counter","Nobody is worthless, and saying immigrants are is hateful.","worthless hateful"),
 (3,"politicians","promote","Politicians are lying scumbags who should all be thrown out.","lying scumbags"),
 (3,"politicians","report","A survey found many voters describe politicians as lying scumbags.","lying scumbags"),
 (3,"politicians","counter","It is lazy and unfair to call all politicians lying scumbags.","lying scumbags"),
 (4,"vegans","promote","Vegans are annoying freaks and nobody likes them.","annoying freaks"),
 (4,"vegans","report","The article quoted a comment calling vegans annoying freaks.","annoying freaks"),
 (4,"vegans","counter","Vegans are not annoying freaks, they just eat differently.","annoying freaks"),
 (5,"teenagers","promote","Teenagers are useless brats who ruin everything.","useless brats"),
 (5,"teenagers","report","The headteacher criticised an advert that called teenagers useless brats.","useless brats"),
 (5,"teenagers","counter","Teenagers are not useless brats and deserve respect.","useless brats"),
 (6,"neighbours","promote","My neighbours are filthy pigs and I hate them.","filthy pigs hate"),
 (6,"neighbours","report","The court heard the defendant called his neighbours filthy pigs.","filthy pigs"),
 (6,"neighbours","counter","It was cruel of him to call his neighbours filthy pigs.","filthy pigs"),
 (7,"refugees","promote","Refugees are parasites draining our country.","parasites"),
 (7,"refugees","report","The report documents politicians describing refugees as parasites.","parasites"),
 (7,"refugees","counter","Refugees are not parasites, and that language dehumanises people.","parasites"),
 (8,"women","promote","Women are too stupid to drive.","stupid"),
 (8,"women","report","Researchers studied online comments claiming women are too stupid to drive.","stupid"),
 (8,"women","counter","The idea that women are too stupid to drive is simply false.","stupid"),
 (9,"poor people","promote","Poor people are lazy losers who deserve nothing.","lazy losers"),
 (9,"poor people","report","The charity condemned a tweet calling poor people lazy losers.","lazy losers"),
 (9,"poor people","counter","Poor people are not lazy losers, and that stereotype hurts.","lazy losers"),
 (10,"referee","promote","That referee is a pathetic moron.","pathetic moron"),
 (10,"referee","report","Fans were fined for chanting that the referee is a pathetic moron.","pathetic moron"),
 (10,"referee","counter","Shouting that the referee is a pathetic moron is unacceptable.","pathetic moron"),
 (11,"gamers","promote","Gamers are smelly idiots.","smelly idiots"),
 (11,"gamers","report","A columnist was criticised for writing that gamers are smelly idiots.","smelly idiots"),
 (11,"gamers","counter","Gamers are not smelly idiots, that is a lazy insult.","smelly idiots"),
 (12,"tourists","promote","Tourists are disgusting and should stay away.","disgusting"),
 (12,"tourists","report","Locals put up signs saying tourists are disgusting, the paper reported.","disgusting"),
 (12,"tourists","counter","It is wrong to say tourists are disgusting.","disgusting"),
 (13,"cyclists","promote","Cyclists are arrogant idiots and should be banned from roads.","arrogant idiots"),
 (13,"cyclists","report","The council received complaints calling cyclists arrogant idiots.","arrogant idiots"),
 (13,"cyclists","counter","Cyclists are not arrogant idiots, most ride safely.","arrogant idiots"),
 (14,"students","promote","Students are lazy scroungers.","lazy scroungers"),
 (14,"students","report","A headline describing students as lazy scroungers sparked complaints.","lazy scroungers"),
 (14,"students","counter","Students are not lazy scroungers, many work two jobs.","lazy scroungers"),
 (15,"rural people","promote","Rural people are dumb hicks.","dumb hicks"),
 (15,"rural people","report","The study found online posts calling rural people dumb hicks.","dumb hicks"),
 (15,"rural people","counter","Rural people are not dumb hicks, that is a nasty stereotype.","dumb hicks"),
 (16,"older people","promote","Old people are useless and a burden.","useless burden"),
 (16,"older people","report","The charity reported that older people are often told they are useless and a burden.","useless burden"),
 (16,"older people","counter","Old people are not useless or a burden.","useless burden"),
]
df = pd.DataFrame(ROWS, columns=["pair_id","topic","condition","text","target_words"])
df["n_words"] = df.text.str.findall(r"\w+").str.len()
print(df.groupby("condition").n_words.mean())   # length check: report sentences are longer (a stated limitation)
df.to_csv(f"{OUT}/exp1_sentences.csv", index=False)

## Cell 3: score every sentence with both filters

In [ ]:
for m in MODELS:
    df[f"p_{m}"] = proba(df.text, m)
print(df.groupby("condition")[["p_original","p_unbiased"]].agg(["mean","std"]).round(3))

## Cell 4: SHAP for every sentence, both models, with the efficiency check

In [ ]:
SV = {}
for m in MODELS:
    SV[m] = shap_explainer(m)(list(df.text), max_evals=500, batch_size=32)
    check_efficiency(SV[m], list(df.text), m)
print("SHAP base value (score of a fully masked sentence, log-odds):",
      {m: round(float(SV[m][0].base_values), 3) for m in MODELS})

## Cell 5: target-word share (the key measure)

In [ ]:
def target_share(e, targets):
    w = shap_word_weights(e); t = set(targets.lower().split())
    pos = {k: v for k, v in w.items() if v > 0}
    return sum(v for k, v in pos.items() if k in t) / max(sum(pos.values()), 1e-9)

for m in MODELS:
    df[f"share_{m}"] = [target_share(SV[m][i], df.target_words[i]) for i in range(len(df))]
print(df.groupby("condition")[["share_original","share_unbiased"]].mean().round(3))

## Cell 6: LIME for every sentence (original model) and SHAP–LIME agreement

In [ ]:
rows, LIME_W = [], []
for i, text in enumerate(df.text):
    lw, exp = lime_word_weights(text, "original")
    LIME_W.append(lw)
    a = agreement(shap_word_weights(SV["original"][i]), lw)
    a["lime_fidelity"] = exp.score          # local fidelity R² of LIME's surrogate (W4-L02 p3)
    rows.append(a)
df = pd.concat([df, pd.DataFrame(rows)], axis=1)
print(df.groupby("condition")[["rho","top1_agree","top3_overlap","lime_fidelity"]].mean().round(3))
df.to_csv(f"{OUT}/exp1_results.csv", index=False)

## Cell 7: Figure 1, the scores chart (slide 6)

In [ ]:
order = ["promote","report","counter"]
g = df.groupby("condition")[["p_original","p_unbiased"]]
mean, std = g.mean().loc[order], g.std().loc[order]
x = np.arange(len(order)); fig, ax = plt.subplots(figsize=(8,5))
for j, m in enumerate(["original","unbiased"]):
    ax.bar(x + (j-0.5)*0.38, mean[f"p_{m}"], 0.38, yerr=std[f"p_{m}"], capsize=4,
           color=COLORS[m], label=f"Detoxify {m}")
ax.set_xticks(x); ax.set_xticklabels(["Promotes abuse","Reports abuse","Argues against abuse"])
ax.set_ylabel("P(toxic)"); ax.set_ylim(0, 1); ax.legend(frameon=False)
ax.set_title("Same insult, different intent: still flagged?")
ax.spines[["top","right"]].set_visible(False); fig.tight_layout()
fig.savefig(f"{OUT}/fig_exp1_scores.png", dpi=200)

## Cell 8: Figure 2, target-word share (slide 6)

In [ ]:
fig, ax = plt.subplots(figsize=(8,5)); rng = np.random.RandomState(SEED)
for j, c in enumerate(order):
    vals = df.loc[df.condition == c, "share_original"]
    ax.scatter(j + rng.uniform(-0.12, 0.12, len(vals)), vals, color=COLORS["original"], alpha=0.7)
    ax.hlines(vals.mean(), j-0.25, j+0.25, color="black", lw=2.5)
ax.set_xticks(range(3)); ax.set_xticklabels(["Promotes","Reports","Argues against"])
ax.set_ylabel("Share of 'toxic' push on insult words"); ax.set_ylim(0, 1.05)
ax.set_title("SHAP: how much of the decision is the insult itself?")
ax.spines[["top","right"]].set_visible(False); fig.tight_layout()
fig.savefig(f"{OUT}/fig_exp1_target_share.png", dpi=200)

## Cell 9: Figures 3 and 4, SHAP vs LIME highlights for the hero sentence (slides 5 and 6)

In [ ]:
i = df.index[(df.pair_id == 1) & (df.condition == "report")][0]
text = df.text[i]
e = SV["original"][i]
shap_pairs = [(tok.strip(), float(v)) for tok, v in zip(e.data, e.values) if tok.strip()]
lime_pairs = [(w, LIME_W[i].get(w.lower(), 0.0)) for w in re.findall(r"\w+", text)]
highlight_png(shap_pairs, f"SHAP  (P(toxic) = {df.p_original[i]:.2f})", f"{OUT}/fig_exp1_hero_shap.png")
highlight_png(lime_pairs, "LIME", f"{OUT}/fig_exp1_hero_lime.png")
with open(f"{OUT}/hero_shap.html", "w") as fh:
    fh.write(shap.plots.text(e, display=False))
for c in order:   # also save SHAP highlights for all three versions of pair 1 (slide 6 option)
    k = df.index[(df.pair_id == 1) & (df.condition == c)][0]; ek = SV["original"][k]
    highlight_png([(t.strip(), float(v)) for t, v in zip(ek.data, ek.values) if t.strip()],
                  f"{c}  (P(toxic) = {df.p_original[k]:.2f})", f"{OUT}/fig_exp1_pair1_{c}.png")

## Cell 10: results summary (numbers to quote on the slides)

In [ ]:
s = df.groupby("condition").agg(p_orig=("p_original","mean"), p_unb=("p_unbiased","mean"),
                                share=("share_original","mean")).loc[order].round(2)
print("=== EXPERIMENT 1 SUMMARY ===")
print(s)
print(f"Report sentences scored > 0.5 toxic (original): {(df[df.condition=='report'].p_original>0.5).sum()}/16")
print(f"Counter sentences scored > 0.5 toxic (original): {(df[df.condition=='counter'].p_original>0.5).sum()}/16")
print(f"SHAP and LIME agreed on the TOP word in {int(df.top1_agree.sum())}/{len(df)} sentences")
print(f"Median SHAP-LIME rank correlation (Spearman rho): {np.nanmedian(df.rho):.2f}")
print(f"Mean top-3 overlap: {df.top3_overlap.mean():.2f}")
print(f"Median LIME local fidelity (R²): {df.lime_fidelity.median():.2f}")
s.to_csv(f"{OUT}/exp1_summary.csv")

## Cell 11: download everything

In [ ]:
!zip -qr outputs_exp1.zip outputs_exp1
from google.colab import files; files.download("outputs_exp1.zip")